In [ ]:
import SimpleITK as sitk  
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
import ipywidgets as widgets
import pandas as pd
from monai.transforms import RemoveSmallObjects, KeepLargestConnectedComponent
import blosc2
from acvl_utils.morphology.morphology_helper import remove_all_but_largest_component

# ============================================================
# Utility Functions
# ============================================================

import numpy as np
import scipy.ndimage as ndi

def interpolate_slices(seg):
    """
    Slice-wise interpolation to repair fully missing slices.
    Only interpolates slices inside the range of annotated slices.

    Args:
        seg (np.ndarray): 3D binary mask (D, H, W)

    Returns:
        np.ndarray: repaired mask
    """
    seg = seg.astype(float)
    repaired = seg.copy()
    D = seg.shape[0]

    # --- Find slice range that contains annotations ---
    slice_sums = np.sum(seg, axis=(1, 2))
    annotated_indices = np.where(slice_sums > 0)[0]

    # If no slices or only one slice is annotated → nothing to interpolate
    if len(annotated_indices) < 2:
        return repaired.astype(np.uint8)

    first_annot = annotated_indices[0]
    last_annot  = annotated_indices[-1]

    # --- Interpolate only *between* annotated slices ---
    for k in range(first_annot + 1, last_annot):
        if slice_sums[k] == 0:   # missing slice
            interpolated = (seg[k-1] + seg[k+1]) / 2
            repaired[k] = (interpolated > 0.5).astype(float)

    return repaired.astype(np.uint8)

def close_segmentation(seg, iterations=2):
    """
    Perform 3D morphological closing to fill small gaps 
    in a 3D segmentation mask.

    Args:
        seg (np.ndarray): 3D binary mask (D, H, W)
        iterations (int): number of dilation/erosion iterations

    Returns:
        np.ndarray: repaired binary mask
    """
    seg = seg.astype(bool)

    # 3D connectivity: 26-connected neighborhood
    structure = ndi.generate_binary_structure(rank=3, connectivity=2)

    # Apply closing
    repaired = ndi.binary_closing(seg, structure=structure, iterations=iterations)

    return repaired.astype(np.uint8)


from scipy.ndimage import distance_transform_edt as dist
def find_annotated_slices(label, axis):
    """Return sorted indices of slices containing label > 0."""
    if axis == "axial":      # slice dimension = 0
        reduce_axes = (1, 2)
    elif axis == "coronal":  # slice dimension = 1
        reduce_axes = (0, 2)
    elif axis == "sagittal": # slice dimension = 2
        reduce_axes = (0, 1)
    else:
        raise ValueError("Invalid axis")

    return np.where(np.any(label > 0, axis=reduce_axes))[0]

def interpolate_missing_slices(label, axis="axial"):
    """
    Detect slices with no annotations and fill them by interpolating
    between the closest annotated slices along the chosen axis.
    """

    lbl = label.copy()
    orig_shape = lbl.shape

    # Move slicing dimension to axis=0 for simplicity
    if axis == "axial":
        lbl = lbl
    elif axis == "coronal":
        lbl = np.transpose(lbl, (1, 0, 2))
    elif axis == "sagittal":
        lbl = np.transpose(lbl, (2, 1, 0))

    D = lbl.shape[0]
    annotated = find_annotated_slices(lbl, "axial")

    if len(annotated) < 2:
        print("Not enough annotated slices to interpolate.")
        return label  # no changes

    # Compute signed distance transform for each slice
    dt = np.zeros_like(lbl, dtype=float)
    for k in range(D):
        fg = lbl[k] > 0
        dt[k] = dist(~fg) - dist(fg)

    # Interpolate only slices with no annotation
    repaired_dt = dt.copy()
    repaired_lbl = lbl.copy()

    for k in range(D):
        if k not in annotated:
            # find nearest annotated slices around k
            prev = annotated[annotated < k]
            next_ = annotated[annotated > k]
            if len(prev) == 0 or len(next_) == 0:
                continue  # cannot interpolate at boundaries

            kp = prev[-1]
            kn = next_[0]

            # linear interpolation in distance-transform space
            w = (k - kp) / float(kn - kp)
            repaired_dt[k] = (1 - w) * dt[kp] + w * dt[kn]
            repaired_lbl[k] = repaired_dt[k] < 0

    # Restore original orientation
    if axis == "axial":
        return repaired_lbl.astype(label.dtype)
    elif axis == "coronal":
        return np.transpose(repaired_lbl, (1, 0, 2)).astype(label.dtype)
    elif axis == "sagittal":
        return np.transpose(repaired_lbl, (2, 1, 0)).astype(label.dtype)

def load_b2nd(path):
    """Load a .b2nd file as a numpy array using blosc2."""
    schunk = blosc2.open(path, mode="r")
    if len(schunk[:])==1:
        arr = schunk[:][0]
    else:
        arr = schunk[:][1]  # Load full array
    return arr

def load_volume(path):
    """Load a 3D volume from NIfTI (.nii.gz), NPZ, NPY, or B2ND files."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix == ".npy":
        return np.load(path)
    elif path.suffix == ".npz":
        data = np.load(path)
        key = list(data.keys())[0]
        return data[key]
          
 
    
    elif path.suffix in [".nii", ".gz"]:
        return  nib.load(str(path)).get_fdata()
    elif path.suffix == ".b2nd":
        return load_b2nd(path)
    else:
        raise ValueError(f"Unsupported format: {path.suffix}")

def resample_label_to_image(label_path, image_ref_path):
    """Resample label to match reference image (CT or input image)."""
    label_sitk = sitk.ReadImage(str(label_path))
    ref_sitk = sitk.ReadImage(str(image_ref_path))
    label_resampled = sitk.Resample(
        label_sitk,
        ref_sitk,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        label_sitk.GetPixelID()
    )
    print(f"Resampled label: {label_resampled.GetSize()} to match CT: {ref_sitk.GetSize()}")
    return sitk.GetArrayFromImage(label_resampled).transpose(2, 1, 0)

def window_ct_hu(ct_hu, level=50, width=350):
    lower, upper = level - width / 2.0, level + width / 2.0
    ct_clipped = np.clip(ct_hu, lower, upper)
    return (ct_clipped - lower) / (upper - lower + 1e-6)

def get_slice(volume, axis, idx):
    if axis == "axial":
        return volume[idx, :, :]
    elif axis == "coronal":
        return volume[:, idx, :]
    elif axis == "sagittal":
        return volume[:, :, idx]
    else:
        raise ValueError(f"Invalid axis: {axis}. Must be 'axial', 'coronal', or 'sagittal'.")

# ============================================================
# Overlay Builders
# ============================================================

def build_label_overlay(label_volume):
    """
    Build a transparent colormap for integer labels (0 = background).
    Supports multi-class visualization.
    """
    unique_labels = np.unique(label_volume)
    print(f"Unique labels found: {unique_labels}")

    # Transparent background
    colors = [[0, 0, 0, 0]]

    # Fixed colors (repeat if more classes)
    predefined_colors = [
        [1, 0, 0, 0.35],  # red
        [0, 1, 0, 0.35],  # green
        [0, 0, 1, 0.35],  # blue
        [1, 1, 0, 0.35],  # yellow
        [1, 0, 1, 0.35],  # magenta
        [0, 1, 1, 0.35],  # cyan
    ]

    for i in range(1, int(unique_labels.max()) + 1):
        colors.append(predefined_colors[(i - 1) % len(predefined_colors)])

    cmap = ListedColormap(colors)
    vmin, vmax = 0, int(unique_labels.max())

    return label_volume.astype(np.int32), cmap, (vmin, vmax)

def build_label_overlay_small_objects_3d(label_volume, min_size=200, connectivity=2):
    delete_small = RemoveSmallObjects(min_size=min_size, connectivity=connectivity)
    cleaned_label = delete_small(label_volume)
    small_objects = (label_volume > 0) & (cleaned_label == 0)
    large_objects = (cleaned_label > 0)
    overlay = np.zeros_like(label_volume, dtype=np.int32)
    overlay[large_objects] = 1
    overlay[small_objects] = 2
    cmap = ListedColormap([[0,0,0,0],[0,1,0,0.25],[1,0,0,0.25]])
    return overlay, cmap, (0, 2)
def build_label_overlay_largest_3d(label_volume):
    """
    Visualize which voxels will be removed when keeping only the largest connected component.
    Red = voxels in the original label
    Green = voxels belonging to the largest component
    Yellow = overlapping (kept) voxels
    """

    # Compute largest connected component mask
    mask_keep = remove_all_but_largest_component(label_volume )

    # Debug info
    print(f"Largest component voxels: {mask_keep.sum()} / {label_volume.sum()} total label voxels")
    #print(mask_keep[180].sum(),mask_keep[:,180,:].sum(), mask_keep[:,:,180].sum() )
    # Build overlay:
    # 0 = background
    # 1 = voxels that will be removed (in label but not in largest component)
    # 2 = voxels that are kept (largest component)
    overlay = np.zeros_like(label_volume, dtype=np.int32)
    overlay[(label_volume > 0) & (~mask_keep)] = 1   # red = removed
    overlay[mask_keep] = 2                           # green = kept

    # Define transparent colormap
    # Red (removed) = [1, 0, 0, 0.35]
    # Green (kept) = [0, 1, 0, 0.35]
    cmap = ListedColormap([
        [0, 0, 0, 0],       # background transparent
        [1, 0, 0, 0.35],    # red = removed
        [0, 1, 0, 0.35],    # green = kept
    ])

    return overlay, cmap, (0, 2)
"""
def build_label_overlay_largest_3d(label_volume):
    largest_component = KeepLargestConnectedComponent(connectivity=1)(label_volume)
    overlay = np.zeros_like(label_volume, dtype=np.int32)
    overlay[label_volume > 0] = 1
    overlay[largest_component > 0] = 2
    print(largest_component.sum(),label_volume.sum() )
    cmap = ListedColormap([[0,0,0,0],[1,0,0,0.25],[0,1,0,0.25]])
    return overlay, cmap, (0, 2)
"""
# ============================================================
# Visualization
# ============================================================

def visualize_case(ct_path, label_path, uid, target,
                   axis="axial", mode="labels",
                   window_level=50, window_width=350,
                   save_dir=None, save_all_slices=False,
                   resample_labels=False):
    """
    Visualize one case with selectable mode:
      - 'labels', 'large_component', 'small_objects'
    Optionally resample labels to match CT grid.
    """
    ct = load_volume(ct_path)
    if resample_labels:
        label = resample_label_to_image(label_path, ct_path)
    else:
        label = load_volume(label_path)

    if not ct_path.suffix == ".b2nd":
        ct = np.transpose(ct, (2, 1, 0)) 
        label = np.transpose(label, (2, 1, 0)) 
        
    print(f"Loaded UID {uid}: CT shape {ct.shape}, Label shape {label.shape}")
    assert ct.shape == label.shape, f"Shape mismatch for {uid}"
    print(np.unique(label))
    if axis == "axial":
        reduce_axes = (1, 2)  
    elif axis == "coronal":
        reduce_axes = (0, 2)  
    elif axis == "sagittal":
        reduce_axes = (0, 1)  

    annotated_slices= np.where(np.any(label > 0, axis=reduce_axes))[0]
    
    if len(annotated_slices) > 0:
        print(f"Annotations at slices: {annotated_slices}")
    else:
        print("No annotations found.")

    #label = interpolate_missing_slices(label, axis=axis)
    #label = interpolate_slices(label)
    #label = close_segmentation(label)

    if mode == "large_component":
        overlay_3d, cmap, vminmax = build_label_overlay_largest_3d(label)
    elif mode == "small_objects":
        overlay_3d, cmap, vminmax = build_label_overlay_small_objects_3d(label)
    else:
        overlay_3d, cmap, vminmax = build_label_overlay(label)

    axis_to_dim = {"axial": 0, "coronal": 1, "sagittal": 2}
    n_slices = ct.shape[axis_to_dim[axis]]

    out_uid_dir = None
    if save_all_slices and save_dir:
        out_uid_dir = Path(save_dir) / str(uid)
        out_uid_dir.mkdir(parents=True, exist_ok=True)

    def plot_slice(idx):
        ct_slice = get_slice(ct, axis, idx)
        overlay_slice = get_slice(overlay_3d, axis, idx)
        ct_img = window_ct_hu(ct_slice, window_level, window_width)

        plt.figure(figsize=(6,6))
        plt.imshow(ct_img, cmap="gray", origin="lower")
        plt.imshow(overlay_slice, cmap=cmap, origin="lower",
                   vmin=vminmax[0], vmax=vminmax[1])
        plt.axis("off")
        plt.title(f"UID {uid} | {target} | {axis.capitalize()} slice {idx}/{n_slices}")
        plt.tight_layout()

        if out_uid_dir:
            plt.savefig(out_uid_dir / f"{axis}_slice{idx:03d}.png", bbox_inches='tight')
            plt.close()
        else:
            plt.show()

    slice_slider = widgets.IntSlider(
        value=n_slices//2, min=0, max=n_slices-1, step=1,
        description=f'UID {uid}', continuous_update=False
    )
    widgets.interact(plot_slice, idx=slice_slider)

    if save_all_slices and out_uid_dir:
        print(f"Saving all slices for UID {uid} → {out_uid_dir}")
        for i in range(n_slices):
            plot_slice(i)

# ============================
# Main Visualization Function
# ============================


def visualize_dataset(uids, img_root, label_root,resample_labels=False,mode="label", axis="axial", save_dir=None, save_all_slices=False):
    """Visualize multiple UIDs."""
    img_root, label_root = Path(img_root), Path(label_root)
    df = pd.read_csv(f"/data/colon_cancer/Classifier/ColonCancer/splits.csv")
    for uid in uids:
        # try naming patterns
        possible_img_names = [
            f"{uid}.nii.gz",f"{uid:03d}.nii.gz", f"{uid}_0000.nii.gz", f"{uid:03d}_0000.nii.gz",
            f"colon_{uid:03d}.nii.gz", f"{uid}.npy", f"{uid}.npz",f"{uid:03d}.npz",  f"{uid:03d}.b2nd",  f"{uid}.b2nd",
        ]
        possible_label_names = [
            f"{uid}.nii.gz",f"{uid:03d}.nii.gz", f"{uid}_0000.nii.gz", f"{uid:03d}_0000.nii.gz",
            f"{uid:03d}_seg.nii.gz",f"{uid}_seg.nii.gz", f"{uid}.npy", f"{uid}_seg.npy",f"{uid:03d}_seg.npz",f"{uid}_seg.npz",  f"{uid:03d}_seg.b2nd", f"{uid}_seg.b2nd"
        ]
        
        for n in possible_img_names:
            print(f"     {img_root / n} -> {'✅ exists' if (img_root / n).exists() else '❌ missing'}")
        for n in possible_label_names:
            print(f"     {label_root / n} -> {'✅ exists' if (label_root / n).exists() else '❌ missing'}")
            
        img_path = next((img_root / n for n in possible_img_names if (img_root / n).exists()), None)
        label_path = next((label_root / n for n in possible_label_names if (label_root / n).exists()), None)
       
        if img_path is None or label_path is None:
            print(f"Skipping UID {uid}: missing image or label")
            continue

        row = df[df["UID"] == uid]
        target = row["target"].values[0]
        if target == 0:
            target = "div"
        else:
            target = "cc"

        visualize_case(img_path, label_path, uid,mode=mode, axis=axis, target=target,
                       save_dir=save_dir, save_all_slices=save_all_slices,resample_labels=resample_labels)


# ============================
# Example usage
# ============================

if __name__ == "__main__":
    #108, 565, 752, 803, 370
    """
    #predictions from validation round
   
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/nnUNet_results/Dataset101_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_0/validation"
    visualize_dataset(uids, img_root, label_root, axis="axial", mode ="label", save_all_slices=False)
 
    #refined labels with resampling
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"
    visualize_dataset(uids, img_root, label_root,axis="axial",mode ="label",resample_labels=True, save_all_slices=False)

    #to be refined labels 
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original"
    visualize_dataset(uids, img_root, label_root,mode ="label",axis="axial", save_all_slices=False)
    
    #original GT
    img_root = "/data/colon_cancer/nnUNet_raw/Dataset100_CC/imagesTr"
    label_root = "/data/colon_cancer/nnUNet_raw/Dataset100_CC/labelsTr"
    visualize_dataset(uids, img_root, label_root,mode ="label", axis="axial", save_all_slices=False)
  
    ##input to segmentator
    img_root = f"/data/colon_cancer/Classifier/ColonCancer/nnUNetPlans_3d_fullres"
    label_root = f"/data/colon_cancer/Classifier/ColonCancer/nnUNetPlans_3d_fullres"
    visualize_dataset(uids, img_root, label_root,mode ="label", axis="axial", save_all_slices=False)
    
    ##input to classifier 
    img_root = f"/data/colon_cancer/CC_Detection/pp_data/Dataset100_CC/rescaledTr"
    label_root = f"/data/colon_cancer/CC_Detection/pp_data/Dataset100_CC/resampledTr/labels_resampled"
    visualize_dataset(uids, img_root, label_root,mode ="label", axis="axial", save_all_slices=False)
  
    #Decathlon predictions
    img_root = "/data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs"
    label_root = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictions_decathlon"
    img_root = "/data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs"
    label_root = "/data/colon_cancer/Classifier/Decathlon/raw_splitted/labelsTs"
    visualize_dataset(uids, img_root, label_root,mode ="label", axis="axial", save_all_slices=False)


    
    
"""
    #99, 74, 158
    #125                 101, 102 
    uids=[18]
  
    img_root = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTs"
    label_root = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTs"
    visualize_dataset(uids, img_root, label_root,mode ="large_component", axis="axial",  save_all_slices=False)
    
    img_root = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTs"
    label_root = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/labelsTs"
    visualize_dataset(uids, img_root, label_root,mode ="label", axis="axial",  save_all_slices=False)
    """
  
    img_root = "/data/colon_cancer/CC_Detection/raw_data/Dataset105_CC/imagesTr"
    label_root = "/data/colon_cancer/CC_Detection/raw_data/Dataset105_CC/labelsTr"
    visualize_dataset(uids, img_root, label_root,mode ="label",axis="axial",  save_all_slices=False)

    img_root = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTr"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"
    visualize_dataset(uids, img_root, label_root,mode ="large_component",axis="axial",resample_labels=True, save_all_slices=False)
    """

In [ ]:
##################################################### check pickle-file generated in nnunet preprocessing steps ############################################################
import pickle
from pathlib import Path 
from tqdm import tqdm
path=f"/data/colon_cancer/nnUNet_preprocessed/Dataset100_CC/nnUNetPlans_3d_fullres/001.pkl"
with open(path, "rb") as f:
    arr = pickle.load(f)
print(arr.keys())
print(arr["shape_after_cropping_and_before_resampling"])
print(arr["shape_before_cropping"])
